In [1]:
import os
import json
import pandas as pd
from maomao.utils.constants import *
from maomao.utils.read_files import load_data_if_nonempty

#### Source to mapping of toxic peptides and organism-level annotation integration
- This notebook augments the curated toxic peptide dataset with organism-level toxicity annotations derived from the original data sources.

- As input, it loads the integrated dataset produced in previous steps, including positive, negative, and ambiguous sequences, together with a curated source-level metadata table that maps each data source to its associated toxicity target (e.g., organism or biological system). All sequences are merged into a unified pivot table where each sequence retains its provenance across multiple sources.

- For each sequence, the notebook identifies which data sources provide annotations by inspecting non-missing entries in the source-specific columns. These source identifiers are then exploded into sequence–source pairs and joined with the external source-to-target mapping, allowing each peptide to be associated with one or more toxicity targets.

- The resulting data are re-aggregated into a sequence-by-target pivot table that summarizes, for each peptide, the presence of evidence across different organism or target categories. Missing associations are explicitly encoded, preserving the distinction between lack of evidence and negative annotation.

- Finally, organism-level statistics are computed and injected into the existing dataset metadata, and the sequence–target association matrix is exported as a standalone artifact. 

In [2]:
toxic_effect = "hemolytic" # Change for different toxic effects (e.g., toxic, neurotoxic, hemolytic, etc.)
integration_folder = f"../../processed_data/integrating_and_cleaning_data"

- Read data

In [3]:
df_organism = (
    pd.read_excel("../../raw_data/tasks_by_source.xlsx")
    .assign(task=lambda x: x["task"].str.lower())
    .loc[lambda x: x["task"].str.contains(f"{toxic_effect}", case=False, na=False)] # Filter data sources by toxic_effect
    .iloc[:, :-4]
)

df_organism

,name source,task,toxicity target
2,BIOPEP-UWM,"celiac toxic, cytotoxic, hemolytic, toxic",human
4,CICERON,"celiac toxic, cytotoxic, hemolytic, embryotoxi...",no information
5,Plantpepdb,"celiac toxic, cytotoxic, hemolytic, toxic, toxins","human, animals"
7,Peptipedia2.0,"cytotoxic, hemolytic, neurotoxic, toxic",no information
8,MultiPep,"celiac toxic, hemolytic, embryotoxic, toxic, i...","human, animals"
24,iAMPCN,"cytotoxic, hemolytic, insecticidal, toxic, ant...",no information
26,Hemolytik2.0_new,hemolytic,"human, animals"
27,AMPDB,"cytotoxic, hemolytic, platelet aggregation inh...",no information
30,Abdelbaky et al.,hemolytic,human
31,AIPAMPDS (HemoRisk-Estimator),hemolytic,human


In [4]:
df_positive = load_data_if_nonempty(key="positive", activity=toxic_effect, filename="positive.csv", path=integration_folder)
df_negative = load_data_if_nonempty(key="negative", activity=toxic_effect, filename="negative.csv", path=integration_folder)
df_ambiguo  = load_data_if_nonempty(key="ambiguous", activity=toxic_effect, filename="ambiguous_data.csv", path=integration_folder)

- Concatenate all dataset

In [5]:
df_pivote = pd.concat([df_negative, df_positive, df_ambiguo])
df_pivote = df_pivote.loc[:, ~df_pivote.columns.str.contains("unlabel")]

In [6]:
df_pivote["sequence"].unique().shape

(33678,)

- Create dataset pivote

In [7]:
# Function to create the 'name source' column with the names of the columns with values ​​other than 999
def create_name_source(row):
    return [col for col in row.index[1:] if row[col] != 999]  # Excluimos la columna 'sequence'

# We apply the function row by row
df_pivote['name source'] = df_pivote.apply(create_name_source, axis=1)

# Expand the 'name source' lists into individual rows
df_exploded = df_pivote.explode('name source').reset_index(drop=True)

df_result = df_exploded[['sequence', 'name source']]

In [8]:
df_exploded = df_exploded.merge(df_organism[['name source', 'toxicity target']], on='name source', how='left')
df_exploded

,sequence,Abdelbaky et al.,Almotairi et al.,AMPDB,AMPDeep,Bhatnagar et al.,BIOPEP-UWM,CICERON,ConsAMPHemo,DRAMP,...,counts_unknown,positive,negative,exclusive_1,exclusive_0,percentage_0,percentage_1,Category_pbb,name source,toxicity target
0,RDRLKDLGSEKIERLRGFQLSGQSDRIRRKILSEFGL,999,999,999,0,999,999,999,999,999,...,31,False,True,False,True,100.0,0.0,NaN,AMPDeep,human
1,RDRLKDLGSEKIERLRGFQLSGQSDRIRRKILSEFGL,999,999,999,0,999,999,999,999,999,...,31,False,True,False,True,100.0,0.0,NaN,counts_1,NaN
2,RDRLKDLGSEKIERLRGFQLSGQSDRIRRKILSEFGL,999,999,999,0,999,999,999,999,999,...,31,False,True,False,True,100.0,0.0,NaN,counts_0,NaN
3,RDRLKDLGSEKIERLRGFQLSGQSDRIRRKILSEFGL,999,999,999,0,999,999,999,999,999,...,31,False,True,False,True,100.0,0.0,NaN,counts_unknown,NaN
4,RDRLKDLGSEKIERLRGFQLSGQSDRIRRKILSEFGL,999,999,999,0,999,999,999,999,999,...,31,False,True,False,True,100.0,0.0,NaN,positive,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
438820,AGKETIFQRLKKKIQEKWKRATIAW,999,999,999,1,999,999,999,999,999,...,28,False,False,False,False,25.0,75.0,70-80,exclusive_1,NaN
438821,AGKETIFQRLKKKIQEKWKRATIAW,999,999,999,1,999,999,999,999,999,...,28,False,False,False,False,25.0,75.0,70-80,exclusive_0,NaN
438822,AGKETIFQRLKKKIQEKWKRATIAW,999,999,999,1,999,999,999,999,999,...,28,False,False,False,False,25.0,75.0,70-80,percentage_0,NaN
438823,AGKETIFQRLKKKIQEKWKRATIAW,999,999,999,1,999,999,999,999,999,...,28,False,False,False,False,25.0,75.0,70-80,percentage_1,NaN


In [9]:
df_exploded["toxicity target"].value_counts()

toxicity target
human             40214
no information    33665
human, animals    25355
Name: count, dtype: int64

In [10]:
df_unique_sequences = df_exploded[['sequence', 'toxicity target']].drop_duplicates()

# Pivot the DataFrame so that the sequences are rows and the toxicity targets are columns
df_pivote_organism = df_unique_sequences.pivot_table(index='sequence', columns='toxicity target', aggfunc='size', fill_value=999)

In [11]:
df_pivote_organism

toxicity target,human,"human, animals",no information
sequence,,,
GAKKGAKKGKKGAKKGAKGAGAKGAGAFKKKK,1,999,999
KRKRAVKRVGRRLKKKLARKIARLGVAF,1,999,999
AAAAAAAAAAGIGKFLHSAKKFGKAFVGEIMNS,1,1,1
AAAAAAAAAGETS,999,999,1
AAAAAAAAAK,1,1,999
...,...,...,...
yPVKLyPVKL,1,1,999
yPVKyPKL,1,1,999
yWfHwK,1,999,999


- Working with metada

In [12]:
with open(f"{integration_folder}/{toxic_effect}/metadata.json", "r") as f:
    metadata = json.load(f)

In [13]:
organism_counts = (
    df_pivote_organism
    .replace(999, 0)
    .sum()
    .astype(int)
    .to_dict()
)

In [14]:
metadata["organism_statistics"] = {"targets": organism_counts}

- Exporting data

In [15]:
with open(f"{integration_folder}/{toxic_effect}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

In [16]:
df_pivote_organism.to_csv(f"{integration_folder}/{toxic_effect}/sequence_by_organism.csv")